# Exercise 09 — Network Resilience

This notebook explores the resilience of the MovieLens user-movie network. Following Lecture 09, we will measure how the network responds to **random failures** versus **targeted attacks**.

## Goal
Quantify the robustness of the recommendation network by simulating the removal of nodes and tracking the size of the Largest Connected Component (LCC). We will compare the effect of losing random users/movies against losing the most connected hubs.

## Setup

Import necessary packages and load the dataset used in previous exercises.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import random

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## Load Data and Build Graph

We use the MovieLens 'small' dataset to build a bipartite graph where nodes are either users or movies, and edges represent ratings.

In [ ]:
ratings = pd.read_csv('../data/movielense/ml-latest-small/ratings.csv')
movies = pd.read_csv('../data/movielense/ml-latest-small/movies.csv')

ratings['user_node'] = ratings['userId'].astype(str).radd('user_')
ratings['movie_node'] = ratings['movieId'].astype(str).radd('movie_')

G = nx.Graph()
G.add_nodes_from(ratings['user_node'].unique(), bipartite=0)
G.add_nodes_from(ratings['movie_node'].unique(), bipartite=1)
G.add_edges_from(zip(ratings['user_node'], ratings['movie_node']))

print(f"Initial graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

## Resilience Simulation Function

We define a function to simulate the removal of nodes. We will track the relative size of the Largest Connected Component (LCC) as the network degrades.

In [ ]:
def simulate_resilience(G, mode='random', steps=50):
    """
    Simulates node removal and returns the fraction of nodes removed and the relative size of the LCC.
    """
    G_sim = G.copy()
    initial_nodes = G_sim.number_of_nodes()
    lcc_sizes = []
    fractions_removed = []
    
    # Initial LCC size
    components = list(nx.connected_components(G_sim))
    if not components:
        return [0], [0]
    
    initial_lcc_size = len(max(components, key=len))
    lcc_sizes.append(1.0)
    fractions_removed.append(0.0)
    
    step_size = initial_nodes // steps
    
    for i in range(steps):
        if G_sim.number_of_nodes() <= step_size:
            break
            
        if mode == 'random':
            to_remove = random.sample(list(G_sim.nodes()), step_size)
        elif mode == 'targeted':
            # Targeted attack on hubs (highest degree nodes)
            # Recalculate degrees for an iterative attack
            degrees = dict(G_sim.degree())
            to_remove = sorted(degrees, key=degrees.get, reverse=True)[:step_size]
        
        G_sim.remove_nodes_from(to_remove)
        
        if G_sim.number_of_nodes() > 0:
            components = list(nx.connected_components(G_sim))
            if components:
                lcc_size = len(max(components, key=len))
                lcc_sizes.append(lcc_size / initial_lcc_size)
            else:
                lcc_sizes.append(0.0)
        else:
            lcc_sizes.append(0.0)
            
        fractions_removed.append((i + 1) * step_size / initial_nodes)
        
    return fractions_removed, lcc_sizes

## Running the Simulations

We run both the random failure and targeted attack simulations. This may take a minute due to repeated LCC calculations.

In [ ]:
print("Starting random failure simulation...")
frac_random, lcc_random = simulate_resilience(G, mode='random')

print("Starting targeted attack simulation...")
frac_targeted, lcc_targeted = simulate_resilience(G, mode='targeted')

print("Simulations complete.")

## Results Visualization

We plot the decay of the LCC for both scenarios to compare the resilience.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(frac_random, lcc_random, label='Random Failure', marker='o', markersize=4)
plt.plot(frac_targeted, lcc_targeted, label='Targeted Attack (Degree Hubs)', marker='s', markersize=4, color='red')

plt.xlabel('Fraction of Nodes Removed')
plt.ylabel('Relative Size of LCC')
plt.title('Network Resilience: MovieLens Bipartite Graph')
plt.legend()
plt.grid(True)
plt.show()

## Interpretation and Conclusion

### 1. Resilience Gap
The simulation results show a striking difference between the two scenarios:
- **Random Failures:** The network exhibits high robustness. The Largest Connected Component degrades slowly, meaning the majority of users and movies stay connected even when many nodes are lost. This is because the vast majority of nodes are low-degree (casual users and niche movies).
- **Targeted Attacks:** The network is extremely vulnerable. Removing just a small percentage of the top hubs (blockbuster movies and highly active critics) causes the LCC to collapse quickly. This is the classic "Achilles' Heel" of scale-free networks, which we confirmed this network to be in Exercise 08.

### 2. Domain Structural Insights
The MovieLens network is held together by a small number of critical hubs. In a recommendation system context, these hubs are the bridges that allow the algorithm to relate different tastes and content. If these "consensus" items or "super-connectors" are removed, the network fragments into isolated islands, making it much harder to generate meaningful recommendations for a broad audience.

### 3. Proposed Intervention
To improve resilience and reduce dependence on a few hubs, the platform should **promote the "Long Tail"**. 

**Policy Change:** Implement a "diversity-aware" recommendation algorithm that explicitly weights mid-degree and low-degree nodes more heavily when building the neighborhood graph. By encouraging users to rate less-known movies and increasing the visibility of niche content, the platform builds a denser, more decentralized network. This redundancy ensures that even if several major hits are removed, there are still enough shared connections between users to maintain a cohesive and functional recommendation environment.